In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from rich.console import Console

from sklearn.datasets import load_breast_cancer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

Visual configuration of Seaborn for more attractive graphics

In [ ]:
sns.set_theme(style="whitegrid", palette="muted")
sns.set_context("notebook", font_scale=1.2)

Rich console instance to display results

In [ ]:
console = Console()

 ## Data Loading

In [ ]:
breast_cancer = load_breast_cancer(as_frame=True)
X = breast_cancer.data
y = breast_cancer.target

target_names = breast_cancer.target_names
diagnosis_labels = breast_cancer.target.map({0: target_names[0], 1: target_names[1]})

console.print(f"Set size: {X.shape}")

## Preprocessing of data

We scale the data using StandardScaler.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## Visualization methods

Method to graph line charts

In [ ]:
def plot_line_chart(X, Y, title, xlabel, ylabel):
    plt.figure(figsize=(8, 4))
    plt.plot(X, Y, marker="o")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()

Method to graph data in 2d

In [ ]:
def plot_data_2d(
    pca, X_pca, y, title, hue_title=None, labels=None, palette="viridis", s=80
):
    hue_values = labels if labels is not None else y

    plt.figure(figsize=(8, 6))
    sns.scatterplot(
        x=X_pca[:, 0],
        y=X_pca[:, 1],
        hue=hue_values,
        palette=palette,
        s=s,
    )

    plt.title(title)
    plt.xlabel(f"PCA 1 ({pca.explained_variance_ratio_[0]:.2%} varianza)")
    plt.ylabel(f"PCA 2 ({pca.explained_variance_ratio_[1]:.2%} varianza)")
    plt.legend(title=hue_title)
    plt.tight_layout()
    plt.show()

Method to apply PCA

In [ ]:
def apply_pca(X_scaled, n_components=None, random_state=42):
    pca = PCA(n_components=n_components, random_state=random_state)
    X_pca = pca.fit_transform(X_scaled)
    return pca, X_pca

## Training Methods

Method to apply k-means

In [ ]:
def apply_kmeans(n_clusters, random_state=42):
    model = KMeans(n_clusters=n_clusters, random_state=random_state)
    return model

Method to get clusters

In [ ]:
def get_clusters(model, X_scaled):
    clusters = model.fit_predict(X_scaled)
    return clusters

Method for obtaining the minimum number of components necessary to achieve minimum variance

In [ ]:
def get_min_components_for_variance(pca, variance):
    cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

    for i, total_var in enumerate(cumulative_variance):
        if total_var >= variance:
            return i, total_var

## 1. K-means Methods

### Elbow Method

In [ ]:
def get_inertia_scores(X_scaled, cluster_range):
    inertia_scores = []

    for k in cluster_range:
        kmeans = apply_kmeans(n_clusters=k)
        kmeans.fit(X_scaled)
        inertia_scores.append(kmeans.inertia_)

    return inertia_scores

In [ ]:
elbow_range = range(1, 11)

intertia = get_inertia_scores(X_scaled, elbow_range)
plot_line_chart(elbow_range, intertia, "Elbow Method", "Number of Clusters", "Inertia")

### Silhouette Method

In [ ]:
def get_silhouette_scores(X_scaled, cluster_range):
    silhouette_scores = []

    for k in cluster_range:
        kmeans = apply_kmeans(n_clusters=k)
        labels = kmeans.fit_predict(X_scaled)
        score = silhouette_score(X_scaled, labels)
        silhouette_scores.append(score)

    return silhouette_scores

In [ ]:
silhouette_range = range(2, 11)

silhouette_scores = get_silhouette_scores(X_scaled, silhouette_range)
plot_line_chart(
    silhouette_range,
    silhouette_scores,
    "Silhouette Method",
    "Number of Clusters",
    "Score",
)

When using the `elbow method`, we observe that starting from 2 clusters, the decrease in inertia slows down, making this number suitable for choosing the number of groups, since adding more clusters does not significantly improve the grouping.

Similarly, with the `silhouette method`, we see that the value 2 is closest to 1, indicating good separation and consistency between the groups. Since both methods agree on this result, it is a good sign that `2 clusters` is an appropriate choice.

## 2. Data visualization in 2D

### Using K-means and PCA

In [ ]:
pca, X_pca = apply_pca(X_scaled, n_components=2)

In [ ]:
kmeans_model = apply_kmeans(n_clusters=2)
clusters = get_clusters(kmeans_model, X_scaled)

plot_data_2d(
    pca, X_pca, clusters, title="Clusters using K-Means and PCA", hue_title="Clusters"
)

### Using Real Diagnostics and PCA

In [ ]:
plot_data_2d(
    pca,
    X_pca,
    y,
    title="Diagnosis Labels with PCA",
    hue_title="Diagnosis Label",
    labels=diagnosis_labels,
)

### Anomalities with PCA

In [ ]:
isolation_forest_model = IsolationForest(contamination=0.05, random_state=42)
isolation_forest_model.fit(X_scaled)
anomalies = isolation_forest_model.predict(X_scaled)
anomaly_label = np.where(anomalies == 1, "Normal", "Anomaly")

In [ ]:
plot_data_2d(
    pca,
    X_pca,
    anomaly_label,
    title="Anomaly Detection with Isolation Forest (PCA)",
    hue_title="Type",
    palette={"Normal": "blue", "Anomaly": "red"},
    s=60,
)

## 3. Questions

### What is the minimum number of principal components needed to explain 95% of the variance?

In [ ]:
pca_full, _ = apply_pca(X_scaled)

n_components, total_var = get_min_components_for_variance(pca_full, 0.95)
print(f"Required components: {n_components}")
print(f"Cumulative variance: {total_var:.2%}")

### What is the variance explained by the first two principal components?

In [ ]:
pca_2d, _ = apply_pca(X_scaled, n_components=2)

total_variance = np.cumsum(pca_2d.explained_variance_ratio_)[-1]

print(f"Cumulative variance (2 components): {total_variance:.2%}")

### Visually inspect the clusters. Could they be easily separated by a linear boundary? (malignant vs. benign)

When grouping the data using K-Means, we can see that the results obtained are quite close to the separation of the actual classes (malignant and benign), indicating that the algorithm was able to identify patterns in the data without prior knowledge of the labels. That said, when projecting this data in two dimensions using PCA, we can see that relatively well-defined groups are formed, so they could be separated by a linear boundary. We could also say that in a logistic regression or a linear SVM, we could achieve a good classification between both categories, thanks to this separation.

## 4. Conclusions

After performing this exercise, I was able to identify how useful it is to apply both the elbow and silhouette methods to determine the number of clusters that best group the data, rather than just sticking with the result of one. In addition, I observed that K-Means works very well in unsupervised training, that is, when there are no labels in the data, as it was able to effectively detect patterns and group the data coherently according to their similarities, this was identified by comparing the graph with and without labels (clusters).